In [32]:
import pandas as pd
import geopandas as gpd
import json


In [34]:

# Define the path to the data file
data_path = r'C:\Users\fasiu\OneDrive\Documents\GitHub\data-layer\Datasets\Raw\Cement Plants\doi_10_5061_dryad_6t1g1jx4f__v20231004\SFI-Global-Cement-Database-assets.csv'

# Read the data file
df = pd.read_csv(data_path)

# Filter for US cement plants
us_plants = df[df.country == 'United States of America']

# Define columns needed for news search
news_search_columns = [
    'uid', 'city', 'state', 'country', 
    'latitude', 'longitude', 'status', 
    'plant_type', 'capacity', 'owner_name', 
    'parent_name', 'raw_mtrl'
]

# Create a filtered dataset
filtered_df = us_plants[news_search_columns].copy()
filtered_df = filtered_df.fillna("")

# Create JSON structure for all plants
all_plants_json = []
for idx, row in filtered_df.iterrows():
    plant_dict = row.to_dict()
    
    # Clean empty strings from JSON
    cleaned_dict = {k: v for k, v in plant_dict.items() if v != ""}
    
    # Add search keywords
    keywords = []
    if plant_dict['city']: keywords.append(plant_dict['city'])
    if plant_dict['state']: keywords.append(plant_dict['state'])
    if plant_dict['owner_name']: keywords.append(plant_dict['owner_name'])
    if plant_dict['parent_name']: keywords.append(plant_dict['parent_name'])
    keywords.extend(['cement', 'cement plant', 'concrete'])
    cleaned_dict['search_keywords'] = keywords
    
    # Create search string
    search_string = f"{plant_dict['city']} {plant_dict['state']} cement plant {plant_dict['owner_name']}".strip()
    cleaned_dict['search_string'] = search_string
    
    all_plants_json.append(cleaned_dict)

dfAllPlants = pd.DataFrame(all_plants_json)

# convert to geopandas
gdf = gpd.GeoDataFrame(dfAllPlants, geometry=gpd.points_from_xy(dfAllPlants.longitude, dfAllPlants.latitude))

# set the coordinate reference system
gdf.crs = {'init': 'epsg:4326'}

# save the geopandas dataframe to a shapefile
gdf.to_file('../data/cement_plants_for_news_search.json', driver='GeoJSON')

c:\Users\fasiu\news_search_facilities\env\Lib\site-packages\pyproj\crs\crs.py:143: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When making the change, be mindful of axis order changes: https://pyproj4.github.io/pyproj/stable/gotchas.html#axis-order-changes-in-proj-6
  in_crs_string = _prepare_from_proj_string(in_crs_string)
